## MEMO – uso della 09_03

**Una volta sola:** pre-tagging, registrazione delle 15 predeterminate (09_02). Non rilanciare.

**Parametri:** `top_j` = chunk all'LLM, `top_k` = candidati del retrieval. Si cambiano nella cella 0 (o `bench.top_j = new`).
Fissare prima del `pre_edit`; non cambiare tra `pre_edit` e `post_edit`.
Dopo averli cambiati usa un `RUN_ID` nuovo (es. `post_edit_j7`) o cancella il vecchio `*_generation.jsonl`: `generate()` salta ciò che è già salvato.

**Ordine di esecuzione**
1. Cella 0 (init), una volta per kernel.
2. Pilota (opz.): 3–4 domande con `bench.generate("pilot", query_ids=[...])`; se C0 è già ~perfetto o molto basso, rivedi `top_j`.
3. Generazione con `RUN_ID = "pre_edit"` (cella 2). Riprendibile: se si interrompe, rilancia.
4. Backup sidecar (poz.): `shutil.copy(sidecar_path, sidecar_path.with_name("dst_benchmark_pre_edit.json"))`.
5. Editing {notebook 09_03a}: cella 1a (cambia `QID`, modifica nel widget), cella 1b per controllare. `bench.edit_status()` conta solo le modifiche query↔chunk delle predq.
6. Generazione con `RUN_ID = "post_edit"` (cella 2).
7. RAGAS (3.1 e 3.2) e report (4): lavorano sui file salvati, non sul sidecar. Imposta `RUN_ID` prima di ogni lancio.

**Altri esperimenti con modifiche diverse**
Cambiare `RUN_ID` rinomina ovviamente solo gli output. Necessario ripartire da una diversa istanziazione del sidecar (es la copia `dst_benchmark_pre_edit.json`); cambiare `sidecar_path` e un `RUN_ID` nuovo.

**Confronto prima/dopo**
Sulle righe C2 e C3 dei due run. C0 non dipende dalle modifiche: nel `post_edit` puoi valutare RAGAS solo su `configs=["C2_dist", "C3_dist_tags"]`.

**Attenzioni**
- `compare_retrieval_vs_reranking` (cella 1b) usa tutti i `top_k` chunk per il baseline, quindi solo per guardare, non per il confronto.
- NaN del giudice riportati e non sostituirli con 0.

In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys, json, logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark

#"""
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)
#"""

QDIR = project_root / "data/queries/dst"
sidecar_path = project_root / "data/processed/dst/dst_benchmark.json"
queries_path = QDIR / "eval_queries.json"          # file unico (42 query): serve a update_query(new_id=...)
pred_path, norm_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json"

handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)

bench = RagasBenchmark(
    handler, predetermined_path=pred_path, normal_path=norm_path,
    out_dir=project_root / "data/benchmark/dst",
    top_j=8, top_k=20, top_n=7,       # stesso numero di chunk (top_j) per TUTTE le configurazioni
    # gold_path=QDIR / "gold_chunks.json",   # opzionale: {query_id: [chunk_id,...]} -> metriche retrieval senza LLM
)
print("!>> Handler e benchmark pronti")

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
13:46:17 [INFO] httpx: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
13:46:20 [INFO] src.pipeline_handler: ?> Pipeline Inizializzata


!>> Handler e benchmark pronti


In [4]:
### PRE-TAGGING ###
candidate_tags = [
    "Orthogonal Representation & Embedding",
    "Bend Minimization & Bounds",
    "Network Flow & Tamassia",
    "Topology-Shape-Metrics Pipeline",
    "Rectilinear Planarity & Complexity",
    "Series-Parallel & Cubic Graphs",
    "Applications & Aesthetics",
]

# per dst_other
#"dense retrieval and retriever (DPR, Contriever, BM25, vector index)",
#"knowledge graph (entities, relations, triples, paths)",
#"graph communities and global sensemaking (community summaries, map-reduce)",
#"hypothetical document generation (HyDE)",
#"noise, distracting documents and context position",
#"self-reflection and critique (reflection tokens, critic, self-critique)",
#"chunking, context window and prompt length",
#"datasets and experimental setup"
#"hallucination and factual grounding"
#"evaluation metrics and win rates (accuracy, nDCG, EM, LLM judge)"
#"baselines and ablation study"
#"prompting and reasoning (prompt template, mind map, chain of thought)"

batch_size = 100

handler.pretag_sidecar(
        sidecar_path=sidecar_path,
        candidate_tags=candidate_tags,
        batch_size=batch_size
)

13:47:12 [INFO] src.pipeline_handler: ?> PreTagging Iniziato.
13:47:12 [INFO] src.pre_tagger: ?>> Inizio pre-tagging sulla collezione 'dst_for_ragas' con 7 tag
13:47:12 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/count "HTTP/1.1 200 OK"
13:47:12 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:47:18 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


DB>> tag: Orthogonal Representation & Embedding
DB>> expanded_tag:
  >>>Orthogonal Representation & Embedding: A technique that involves the use of orthogonal matrices to represent and embed high-dimensional data into lower-dimensional spaces, often utilizing tools like PCA, t-SNE, or Autoencoders, while related entities include dimensionality reduction, feature extraction, manifold learning, and neural networks.


13:47:35 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


DB>> tag: Bend Minimization & Bounds
DB>> expanded_tag:
  >>>Bend Minimization & Bounds: A computational technique used in molecular modeling and structural biology to minimize the energy of a molecule by adjusting its bond angles, involving tools like molecular mechanics and dynamics simulations, and related entities such as dihedral angles, torsional potentials, and force fields.


13:47:50 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


...

13:50:56 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


DB>> tag: Series-Parallel & Cubic Graphs
DB>> expanded_tag:
  >>>Series-Parallel & Cubic Graphs: rappresentazioni grafiche serie-parallele e cubiche che includono sotto-categorie principali come Grafici Serie-Paralleli, Grafici Cubici, Strumenti di disegno come Pencil, Calcolatori, Software di visualizzazione dati, ed entità o termini tecnici correlati come Topologia dei grafici, Teoria dei grafi, Algebra lineare.


13:51:16 [INFO] httpx: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Pre-tagging 'dst_for_ragas':   1%|          | 26/2120 [04:27<5:56:24, 10.21s/chunk]

DB>> tag: Applications & Aesthetics
DB>> expanded_tag:
  >>>Applications & Aesthetics: campo interdisciplinare che combina l'analisi delle applicazioni e dell'estetica, connessa a discipline come la design sostenibile, l'ergonomia, le tecnologie emergenti, i materiali innovativi, gli ambienti di lavoro collaborativi, le esperienze immersive, il branding digitale, la comunicazione visiva, la fotografia artistica, la grafica interattiva, la progettazione dei prodotti, la progettazione urbana sostenibile, l'illuminotecnica, i materiali ecocompatibili, le tecnologie di realtà aumentata e virtuale.


13:51:41 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:42 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:42 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:43 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:44 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:45 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:46 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:47 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
13:51:47 [INFO] httpx: H

In [ ]:
### CONFIG PREDQ ### (se non gia' fatto in 09_02) + controllo similarita' per eventuali query in più sopra threshold
#bench.register_predetermined()
print(bench.check_similarity().head(5))

In [ ]:
### PILOTA ### domande x (top_j = 8 e 10): serve a vedere se C0 è già al soffitto o troppo basso
import re, pandas as pd
PILOT_IDS = ["CPX_P01", "ALG_P03", "VIN_P02", "CMP_P02", "DEF_N03", "CPX_N01", "CHI_N01"] 

def coverage(ref, ctxs):
    """Proxy gratuito del context recall: quota di parole 'forti' della reference presenti nei chunk."""
    words = set(re.findall(r"[a-z0-9.]{5,}", ref.lower()))
    text = " ".join(ctxs).lower()
    return sum(w in text for w in words) / max(len(words), 1)

rows = []
for j in (8, 10):
    bench.top_j = j
    g = bench.generate(f"pilot_j{j}", query_ids=PILOT_IDS)      # run_id diverso per ogni j
    g["top_j"] = j
    g["coverage"] = [coverage(r, c) for r, c in zip(g.reference, g.retrieved_contexts)]
    rows.append(g)
pil = pd.concat(rows)

# 1) copertura media per configurazione e top_j (soffitto ~1.0 = poco margine; molto bassa = serve più j)
display(pil.pivot_table(index="config", columns="top_j", values="coverage", aggfunc="mean").round(2))
# 2) per domanda
display(pil.pivot_table(index=["query_id", "config"], columns="top_j", values="coverage").round(2))
# 3) leggi a mano 2-3 risposte baseline per capire se llama3.2 risponde in modo sensato
for _, r in pil[(pil.config == "C0_baseline") & (pil.top_j == 5)].iterrows():
    print(f"\n[{r.query_id}] {r.user_input}\n  RISPOSTA: {r.response[:400]}\n  REFERENCE: {r.reference[:300]}")
print("\nsecondi medi per risposta:", pil.groupby("top_j").seconds.mean().round(1).to_dict())

In [ ]:
### 2 ### GENERAZIONE (Ollama) - riprendibile: se si interrompe basta rilanciare la cella
RUN_ID = "pre_edit"          # poi "post_edit" dopo i ritocchi manuali
gen = bench.generate(RUN_ID)
print(gen.groupby("config").seconds.mean().round(1))
print("predq non riconosciute/errate:", gen[~gen.predq_ok].query_id.unique())
# Metriche di retrieval senza giudice (solo se gold_path impostato)
print(bench.retrieval_summary(RUN_ID))

In [ ]:
### 3.1 ### VALUTAZIONE RAGAS con giudice esterno (piu' forte di llama3.2) - una configurazione alla volta, riprendibile
# Chiavi API da variabili d'ambiente (mai nel notebook). Esempi (verifica il nome del modello attuale):
from langchain_google_genai import ChatGoogleGenerativeAI      # pip install langchain-google-genai
judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)   # GOOGLE_API_KEY
# from langchain_groq import ChatGroq; judge_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Prova prima su 1 configurazione per stimare token e tempo; max_workers basso per i rate limit del free tier
ragas_df = bench.evaluate_ragas(RUN_ID, judge_llm, max_workers=2, configs=["C0_baseline"])

In [ ]:
### 3.2 ### Poi tutte le configurazioni
ragas_df = bench.evaluate_ragas(RUN_ID, judge_llm, max_workers=2)
print(ragas_df.isna().sum())      # NaN = fallimenti del giudice: riportali nella tesi, non sostituirli con 0

In [ ]:
### 4 ### REPORT per la tesi
ragas_df = bench.load_ragas(RUN_ID)
display(bench.summary(ragas_df, by="category"))          # per categoria (6 sezioni del professore)
display(bench.summary(ragas_df, by="is_predetermined"))  # in-sample (ritoccate) vs generalizzazione
display(bench.paired_vs_baseline(ragas_df))              # tutte le query
display(bench.paired_vs_baseline(ragas_df, subset=~ragas_df.is_predetermined))   # solo normali (generalizzazione)
ragas_df.to_csv(bench.out_dir / f"{RUN_ID}_ragas_all.csv", index=False)